In [0]:
%run ./00_config_and_logging

In [0]:
from pyspark.sql.functions import col, explode_outer, split, countDistinct

# Feasibility check: do the 10 synthetic assets' software products have real
# CVEs with CPE matches in our NVD data? Uses a 25% sample of silver's clean
# CVE universe (seed=42, reproducible) to keep it fast.

# software_name -> CPE vendor:product candidates (cpe:2.3:a:<vendor>:<product>:...)
software_cpe = [
    ("Apache HTTP Server", ["apache:http_server"]),
    ("Microsoft SQL Server", ["microsoft:sql_server"]),
    ("OpenSSH", ["openbsd:openssh"]),
    ("Redis", ["redis:redis"]),
    ("Nginx", ["nginx:nginx", "f5:nginx"]),
    ("Prometheus", ["prometheus:prometheus"]),
    ("Samba", ["samba:samba"]),
    ("Docker", ["docker:docker"]),
    ("OpenVPN", ["openvpn:openvpn"]),
    ("Logstash", ["elastic:logstash", "logstash:logstash"]),
]

pair_to_software = {}
cond = None
for software, cps in software_cpe:
    for cp in cps:
        v, p = cp.split(":")
        pair_to_software[(v, p)] = software
        c = (col("vendor") == v) & (col("product") == p)
        cond = c if cond is None else cond | c

# 1) 25% sample of silver's clean CVE universe (product-level check, so we
#    only need the cve_id; CPE criteria live in bronze, not silver)
sampled_ids = (spark.table(SILVER_UNIFIED).select("cve_id").distinct()
               .sample(withReplacement=False, fraction=0.25, seed=42))
print("Silver CVE sample (25%):", sampled_ids.count())

# 2) explode CPE criteria from bronze rows for the sampled CVEs only
# (bronze_nvd stores the CVE id in `id`; alias it to cve_id for the join)
cpe = (spark.table(BRONZE_NVD).select(col("id").alias("cve_id"), "configurations")
       .join(sampled_ids, on="cve_id", how="left_semi")
       .select("cve_id", explode_outer("configurations").alias("conf"))
       .select("cve_id", explode_outer("conf.nodes").alias("node"))
       .select("cve_id", explode_outer("node.cpeMatch").alias("cm"))
       .where(col("cm.criteria").isNotNull())
       .select("cve_id", split(col("cm.criteria"), ":").alias("parts"))
       .select("cve_id", col("parts")[3].alias("vendor"), col("parts")[4].alias("product")))

matched = cpe.where(cond)

res = matched.groupBy("vendor", "product").agg(countDistinct("cve_id").alias("cves")).collect()
print("\nProduct-level matches (version ranges NOT yet applied):")
for r in sorted(res, key=lambda r: -r["cves"]):
    sw = pair_to_software.get((r["vendor"], r["product"]), "?")
    print(f"  {sw:22s} ({r['vendor']}:{r['product']}): {r['cves']} CVEs")

covered = {pair_to_software.get((r["vendor"], r["product"])) for r in res}
print(f"\nSoftware with >=1 CVE match: {len(covered)}/10 -> {sorted(covered)}")
print("Total distinct CVEs matching any product:", matched.select("cve_id").distinct().count())

In [0]:
# Logstash gap check: newer NVD records put product info in `affected` instead
# of CPE configurations. Count sampled CVEs matching elastic:logstash there.
sampled_ids = spark.table(SILVER_UNIFIED).select("cve_id").distinct() \
    .sample(withReplacement=False, fraction=0.25, seed=42)

affected_match = (spark.table(BRONZE_NVD)
    .select(col("id").alias("cve_id"), "affected")
    .join(sampled_ids, on="cve_id", how="left_semi")
    .select("cve_id", explode_outer("affected").alias("aff"))
    .select("cve_id", explode_outer("aff.affectedData").alias("ad"))
    .where((col("ad.vendor") == "elastic") & (col("ad.product") == "logstash"))
    .select("cve_id").distinct())

n = affected_match.count()
print(f"CVEs with affected=elastic:logstash in 25% sample: {n}")
if n > 0:
    affected_match.orderBy("cve_id").show(n, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
)
from datetime import datetime
import re

GOLD_RUN_ID = new_run_id()

# ---- Reusable idempotent MERGE loader (03a pattern, extended) ----
# Creates the table if missing; if an older overwrite-based version of the
# table exists (no record_hash column), rebuilds it once under the new schema.
# The MERGE itself is idempotent: record_hash prevents no-op updates.
def merge_dimension(source_df, target_table, key_condition, view_name):
    """Creates the table if it does not exist, then runs an idempotent MERGE."""
    if spark.catalog.tableExists(target_table):
        cols = [c.name for c in spark.table(target_table).schema.fields]
        if "record_hash" not in cols:
            # legacy schema from the earlier overwrite-based build -> rebuild once
            source_df.limit(0).write.format("delta") \
                .mode("overwrite").option("overwriteSchema", "true") \
                .saveAsTable(target_table)
            print(f"Rebuilt {target_table} under the new schema (legacy table replaced)")
    else:
        source_df.limit(0).write.format("delta") \
            .mode("ignore") \
            .saveAsTable(target_table)

    source_df.createOrReplaceTempView(view_name)

    spark.sql(f"""
        MERGE INTO {target_table} t
        USING {view_name} s
        ON {key_condition}

        WHEN MATCHED AND NOT (t.record_hash <=> s.record_hash)
            THEN UPDATE SET *

        WHEN NOT MATCHED
            THEN INSERT *
    """)

    metrics = (
        spark.sql(f"DESCRIBE HISTORY {target_table}")
        .filter("operation = 'MERGE'")
        .orderBy(F.col("version").desc())
        .first()["operationMetrics"]
    )
    return (int(metrics.get("numTargetRowsInserted", 0)),
            int(metrics.get("numTargetRowsUpdated", 0)))

print("Gold pipeline setup complete. Run ID:", GOLD_RUN_ID)

In [0]:
# ---- dim_severity: CVSS bands (03a design) ----
t0 = datetime.now()
severity_data = [
    (0, "UNKNOWN", 0, None, None),
    (1, "LOW", 1, 0.1, 3.9),
    (2, "MEDIUM", 2, 4.0, 6.9),
    (3, "HIGH", 3, 7.0, 8.9),
    (4, "CRITICAL", 4, 9.0, 10.0),
]
severity_schema = StructType([
    StructField("severity_key", IntegerType(), False),
    StructField("severity_name", StringType(), False),
    StructField("severity_rank", IntegerType(), False),
    StructField("cvss_score_min", DoubleType(), True),
    StructField("cvss_score_max", DoubleType(), True),
])

dim_severity = (
    spark.createDataFrame(severity_data, severity_schema)
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("severity_key").cast("string"),
                F.col("severity_name"),
                F.col("severity_rank").cast("string"),
                F.coalesce(F.col("cvss_score_min").cast("string"), F.lit("")),
                F.coalesce(F.col("cvss_score_max").cast("string"), F.lit("")),
            ),
            256,
        ),
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

ins, upd = merge_dimension(dim_severity, GOLD_DIM_SEVERITY,
                           "t.severity_key = s.severity_key",
                           f"dim_severity_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", "static_reference", GOLD_DIM_SEVERITY, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_dim_severity | inserted={ins}, updated={upd}")

In [0]:
# ---- dim_date: calendar over silver's full date surface (03a design) ----
# Covers published, last_modified, kev_date_added and kev_due_date.
t0 = datetime.now()
date_bounds = (
    spark.table(SILVER_UNIFIED)
    .select(
        F.to_date("published_date").alias("published_date"),
        F.to_date("last_modified").alias("last_modified_date"),
        F.to_date("kev_date_added").alias("kev_date_added_date"),
        F.to_date("kev_due_date").alias("kev_due_date_date"),
    )
    .select(
        F.least("published_date", "last_modified_date", "kev_date_added_date", "kev_due_date_date").alias("possible_min_date"),
        F.greatest("published_date", "last_modified_date", "kev_date_added_date", "kev_due_date_date").alias("possible_max_date"),
    )
    .agg(
        F.min("possible_min_date").alias("min_date"),
        F.max("possible_max_date").alias("max_date"),
    )
)

dim_date = (
    spark.range(1)
    .crossJoin(date_bounds)
    .select(F.explode(F.expr("sequence(min_date, max_date, interval 1 day)")).alias("full_date"))
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter_number", F.quarter("full_date"))
    .withColumn("year_number", F.year("full_date"))
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("date_key").cast("string"),
                F.col("full_date").cast("string"),
                F.col("day_name"),
                F.col("month_name"),
                F.col("quarter_number").cast("string"),
                F.col("year_number").cast("string"),
            ),
            256,
        ),
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

ins, upd = merge_dimension(dim_date, GOLD_DIM_DATE,
                           "t.date_key = s.date_key",
                           f"dim_date_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", SILVER_UNIFIED, GOLD_DIM_DATE, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_dim_date | inserted={ins}, updated={upd}")

In [0]:
# ---- dim_cve: surrogate cve_key + rich descriptive columns (both designs) ----
t0 = datetime.now()
dim_cve = (
    spark.table(SILVER_UNIFIED)
    .filter(F.col("cve_id").isNotNull() & (F.trim(F.col("cve_id")) != ""))
    .select(
        F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"),
        "cve_id", "description", "cwe_ids", "vuln_status",
        "cvss_severity", "cvss_vector",
        "published_date", "last_modified", "source_identifier",
        F.col("vendor_project").alias("kev_vendor_project"),
        F.col("product").alias("kev_product"),
        F.col("vulnerability_name").alias("kev_vulnerability_name"),
        F.col("short_description").alias("kev_description"),
        F.col("required_action").alias("kev_required_action"),
        "known_ransomware_use", "kev_date_added", "kev_due_date",
    )
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("cve_id"), F.lit("")),
                F.coalesce(F.col("description"), F.lit("")),
                F.coalesce(F.concat_ws(",", F.col("cwe_ids")), F.lit("")),
                F.coalesce(F.col("vuln_status"), F.lit("")),
                F.coalesce(F.col("cvss_severity"), F.lit("")),
                F.coalesce(F.col("cvss_vector"), F.lit("")),
                F.coalesce(F.col("published_date").cast("string"), F.lit("")),
                F.coalesce(F.col("last_modified").cast("string"), F.lit("")),
                F.coalesce(F.col("kev_date_added").cast("string"), F.lit("")),
                F.coalesce(F.col("kev_due_date").cast("string"), F.lit("")),
                F.coalesce(F.col("known_ransomware_use"), F.lit("")),
            ),
            256,
        ),
    )
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

ins, upd = merge_dimension(dim_cve, GOLD_DIM_CVE,
                           "t.cve_key = s.cve_key",
                           f"dim_cve_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", SILVER_UNIFIED, GOLD_DIM_CVE, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_dim_cve | inserted={ins}, updated={upd}")

In [0]:
# ---- dim_asset: asset inventory with change detection (our design + 03a hashing) ----
# Bronze is append-only, so the same asset can arrive multiple times (e.g. a
# Mockaroo re-send or regenerated record). Keep only the LATEST row per
# asset_id (by load_timestamp) so the pick is deterministic, and the record_hash
# MERGE makes identical re-sends a no-op.
from pyspark.sql import Window
t0 = datetime.now()
_asset_cols = ["asset_id", "hostname", "ip_address", "os_name", "os_version",
               "software_name", "software_version", "department", "owner",
               "criticality", "exposure", "environment", "location", "status"]
dim_asset = (
    spark.table(BRONZE_ASSETS)
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("asset_id").orderBy(F.col("load_timestamp").desc())))
    .where(F.col("rn") == 1)
    .drop("rn")
    .select(*_asset_cols)
    .withColumn("record_hash",
                F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("")) for c in _asset_cols]), 256))
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

ins, upd = merge_dimension(dim_asset, GOLD_DIM_ASSET,
                           "t.asset_id = s.asset_id",
                           f"dim_asset_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", BRONZE_ASSETS, GOLD_DIM_ASSET, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_dim_asset | inserted={ins}, updated={upd}")

In [0]:
# ---- dim_software: software_name -> CPE vendor:product mapping (our design) ----
# The bridge between the asset inventory and NVD's CPE match criteria.
# Covers BOTH the original 10-asset feed (still in bronze: OpenSSH, Nginx, Samba)
# and the new Mockaroo software list (Kubernetes, MySQL, Node.js, PostgreSQL,
# Tomcat). Nginx has two CPE spellings (nginx:nginx, f5:nginx) -> two rows.
# MySQL: NVD uses oracle:mysql (mysql:mysql has zero matches in our NVD data).
t0 = datetime.now()
software_cpe = [
    ("Apache HTTP Server",   "apache",      "http_server"),
    ("Microsoft SQL Server", "microsoft",   "sql_server"),
    ("OpenSSH",              "openbsd",     "openssh"),
    ("Redis",               "redis",        "redis"),
    ("Nginx",               "nginx",        "nginx"),
    ("Nginx",               "f5",           "nginx"),
    ("Prometheus",           "prometheus",  "prometheus"),
    ("Samba",               "samba",        "samba"),
    ("Docker",              "docker",       "docker"),
    ("OpenVPN",              "openvpn",      "openvpn"),
    ("Logstash",             "elastic",      "logstash"),
    ("Kubernetes",           "kubernetes",   "kubernetes"),
    ("MySQL",                "oracle",       "mysql"),
    ("Node.js",              "nodejs",      "node.js"),
    ("PostgreSQL",           "postgresql",  "postgresql"),
    ("Tomcat",               "apache",      "tomcat"),
]

dim_software = (
    spark.createDataFrame(software_cpe, ["software_name", "cpe_vendor", "cpe_product"])
    .withColumn("software_key", F.sha2(F.concat_ws(":", "software_name", "cpe_vendor", "cpe_product"), 256))
    .withColumn("record_hash", F.sha2(F.concat_ws("||", "software_name", "cpe_vendor", "cpe_product"), 256))
    .withColumn("gold_load_timestamp", F.current_timestamp())
)

ins, upd = merge_dimension(dim_software, GOLD_DIM_SOFTWARE,
                           "t.software_key = s.software_key",
                           f"dim_software_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", "software_cpe_map", GOLD_DIM_SOFTWARE, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_dim_software | inserted={ins}, updated={upd}")

In [0]:
from pyspark.sql.functions import udf

# ---- version-range comparator (CPE bounds use dotted versions; asset versions
# look like '2.4.52', '9.6p1', '2019') ----
def _parse(v):
    if v is None:
        return []
    parts = []
    for seg in str(v).split("."):
        m = re.match(r"^(\d+)", seg.strip())
        parts.append((int(m.group(1)) if m else 0, seg.strip()))
    return parts

def _cmp(a, b):
    n = max(len(a), len(b))
    a = a + [(0, "")] * (n - len(a))
    b = b + [(0, "")] * (n - len(b))
    for sa, sb in zip(a, b):
        if sa != sb:
            return -1 if sa < sb else 1
    return 0

def version_matches(asset_ver, cpe_ver, s_exc, s_inc, e_exc, e_inc):
    """True if the asset's version satisfies the CPE match criteria bounds."""
    if asset_ver is None:
        return False
    a = _parse(asset_ver)
    if cpe_ver not in (None, "", "*") and _cmp(a, _parse(cpe_ver)) != 0:
        return False
    if s_exc and not _cmp(a, _parse(s_exc)) > 0:
        return False
    if s_inc and _cmp(a, _parse(s_inc)) < 0:
        return False
    if e_exc and not _cmp(a, _parse(e_exc)) < 0:
        return False
    if e_inc and _cmp(a, _parse(e_inc)) > 0:
        return False
    return True

version_matches_udf = udf(version_matches, BooleanType())

# ---- 1) explode CPE criteria from bronze NVD (one row per cve x cpeMatch) ----
t0 = datetime.now()
cpe_rows = (spark.table(BRONZE_NVD)
    .select(F.col("id").alias("cve_id"), "configurations")
    .dropDuplicates(["cve_id"])
    .select("cve_id", F.explode_outer("configurations").alias("conf"))
    .select("cve_id", F.explode_outer("conf.nodes").alias("node"))
    .select("cve_id", F.explode_outer("node.cpeMatch").alias("cm"))
    .where(F.col("cm.criteria").isNotNull() & (F.col("cm.vulnerable") == True))
    .select("cve_id",
            F.split(F.col("cm.criteria"), ":").alias("parts"),
            F.col("cm.versionStartExcluding").alias("vs_exc"),
            F.col("cm.versionStartIncluding").alias("vs_inc"),
            F.col("cm.versionEndExcluding").alias("ve_exc"),
            F.col("cm.versionEndIncluding").alias("ve_inc"))
    .select("cve_id",
            F.col("parts")[3].alias("cpe_vendor"),
            F.col("parts")[4].alias("cpe_product"),
            F.col("parts")[5].alias("cpe_version"),
            "vs_exc", "vs_inc", "ve_exc", "ve_inc"))

# ---- 2) match software + asset, version check, join silver measures, map keys ----
fact = (cpe_rows
    .join(spark.table(GOLD_DIM_SOFTWARE).select("software_key", "software_name", "cpe_vendor", "cpe_product"),
          on=["cpe_vendor", "cpe_product"])
    .join(spark.table(GOLD_DIM_ASSET).select("asset_id", "software_name", "software_version"),
          on="software_name")
    .where(version_matches_udf(F.col("software_version"), F.col("cpe_version"),
                               F.col("vs_exc"), F.col("vs_inc"), F.col("ve_exc"), F.col("ve_inc")))
    .join(spark.table(SILVER_UNIFIED).select(
            "cve_id", "published_date", "kev_date_added",
            "cvss_base_score", "epss_score", "epss_percentile", "is_in_kev"),
          on="cve_id")
    .select(
        "asset_id", "cve_id",
        F.sha2(F.trim(F.col("cve_id")), 256).alias("cve_key"),
        "software_key",
        F.when(F.col("cvss_base_score") >= 9.0, F.lit(4))
         .when(F.col("cvss_base_score") >= 7.0, F.lit(3))
         .when(F.col("cvss_base_score") >= 4.0, F.lit(2))
         .when(F.col("cvss_base_score") >= 0.1, F.lit(1))
         .otherwise(F.lit(0)).alias("severity_key"),
        F.date_format("published_date", "yyyyMMdd").cast("int").alias("published_date_key"),
        F.when(F.col("kev_date_added").isNotNull(),
               F.date_format("kev_date_added", "yyyyMMdd").cast("int")).alias("kev_date_key"),
        "cvss_base_score", "epss_score", "epss_percentile", "is_in_kev",
        F.when(F.col("is_in_kev") == True, F.lit(1))
         .when(F.col("epss_score") >= 0.5, F.lit(2))
         .when(F.col("cvss_base_score") >= 9.0, F.lit(3))
         .otherwise(F.lit(4)).alias("priority_rank"))
    .dropDuplicates(["asset_id", "cve_key"])
    .withColumn("record_hash",
                F.sha2(F.concat_ws("||",
                    F.col("asset_id"), F.col("cve_key"),
                    F.coalesce(F.col("cvss_base_score").cast("string"), F.lit("")),
                    F.coalesce(F.col("epss_score").cast("string"), F.lit("")),
                    F.coalesce(F.col("is_in_kev").cast("string"), F.lit(""))), 256))
    .withColumn("gold_load_timestamp", F.current_timestamp()))

ins, upd = merge_dimension(fact, GOLD_FACT_VULN,
                           "t.asset_id = s.asset_id AND t.cve_key = s.cve_key",
                           f"fact_src_{GOLD_RUN_ID.replace('-', '')}")
write_log(GOLD_RUN_ID, "Silver-to-Gold", "merge", f"{BRONZE_NVD}+{SILVER_UNIFIED}", GOLD_FACT_VULN, t0,
          "Success", rows_inserted=ins, rows_updated=upd)
print(f"gold_fact_vulnerability | inserted={ins}, updated={upd}")

In [0]:
# ---- Gold build verification: counts, FK integrity, business views ----
fact_tbl = spark.table(GOLD_FACT_VULN)
print("Fact rows (asset x cve):", fact_tbl.count())
print("Distinct CVEs:", fact_tbl.select("cve_key").distinct().count())
print("Assets covered:", fact_tbl.select("asset_id").distinct().count())

# FK integrity: every fact key must resolve in its dimension
orphan_cve  = fact_tbl.join(spark.table(GOLD_DIM_CVE).select("cve_key"), on="cve_key", how="left_anti").count()
orphan_asset = fact_tbl.join(spark.table(GOLD_DIM_ASSET).select("asset_id"), on="asset_id", how="left_anti").count()
orphan_sev  = fact_tbl.join(spark.table(GOLD_DIM_SEVERITY).select("severity_key"), on="severity_key", how="left_anti").count()
orphan_date = (fact_tbl.select(F.col("published_date_key").alias("dk"))
    .join(spark.table(GOLD_DIM_DATE).select(F.col("date_key").alias("dk")), on="dk", how="left_anti").count())
print(f"Orphan FKs -> cve: {orphan_cve} | asset: {orphan_asset} | severity: {orphan_sev} | date: {orphan_date}")

print("\nExposures per asset:")
fact_tbl.join(spark.table(GOLD_DIM_ASSET), "asset_id") \
    .groupBy("asset_id", "hostname", "software_name") \
    .agg(F.count("*").alias("cves"),
         F.max("cvss_base_score").alias("max_cvss"),
         F.max("epss_score").alias("max_epss"),
         F.max(F.col("is_in_kev").cast("int")).alias("any_kev")) \
    .orderBy(F.col("cves").desc()).show()

print("Severity mix (fact rows per severity):")
fact_tbl.join(spark.table(GOLD_DIM_SEVERITY), "severity_key") \
    .groupBy("severity_name").agg(F.count("*").alias("fact_rows")) \
    .orderBy("severity_name").show()

print("KEV-exposed asset pairs (fix first):")
fact_tbl.where(F.col("is_in_kev") == True) \
    .join(spark.table(GOLD_DIM_ASSET), "asset_id") \
    .select("hostname", "cve_id", "cvss_base_score", "epss_score").show()